# 📊 Electricity Demand & Supply Forecasting - Uncertainty Quantification & Confidence Intervals

This notebook calculates **90%, 95%, and 99% prediction/confidence intervals** for both **Demand Forecasting** and **Supply Forecasting** models, as well as the **Demand–Supply Gap** for **January, February, and March 2026**.

### Statistical Methodology:
1. **Residual Standard Errors (sigma_e)**: Quantified from model validation residuals up to December 2025 (strictly zero data leakage for 2026).
   - **Demand Model Standard Error (sigma_D)**: `391.30 MU`
   - **Supply Model Standard Error (sigma_S)**: `996.83 MU`
   - **Combined Gap Standard Error (sigma_Gap)**: `1070.88 MU`  
2. **Confidence Intervals**: Centered on point forecast using standard normal multipliers ($z_{0.95} = 1.645$, $z_{0.975} = 1.960$, $z_{0.995} = 2.576$):
   $$\text{PI}_{1-\alpha} = \left[ \hat{y} - z_{1-\alpha/2} \cdot \sigma_e, \quad \hat{y} + z_{1-\alpha/2} \cdot \sigma_e \right]$$
3. **Gap Classification**: $\text{Gap} = \text{Demand} - \text{Supply}$. Classified as **Shortage** (Demand > Supply), **Surplus** (Supply > Demand), or **Balanced**.

In [1]:
# ============================================================
# STEP 1: LOAD TRAINED MODELS & SCALERS
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

demand_dir = r'../Electricity Demand and Supply Forecasting/Model Training/Demand forecasting'
supply_dir = r'../Electricity Demand and Supply Forecasting/Model Training/Supply Forecasting'
val_dir = r'../Electricity Demand and Supply Forecasting/Model Training/Demand_Forecasting_Validation'

d_model = load_model(os.path.join(demand_dir, 'Demand_LSTM_Final.keras'))
d_X_scaler = joblib.load(os.path.join(demand_dir, 'Demand_LSTM_X_scaler.pkl'))
d_y_scaler = joblib.load(os.path.join(demand_dir, 'Demand_LSTM_y_scaler.pkl'))

s_model = load_model(os.path.join(supply_dir, 'Supply_LSTM.keras'))
s_X_scaler = joblib.load(os.path.join(supply_dir, 'Supply_LSTM_X_scaler.pkl'))
s_y_scaler = joblib.load(os.path.join(supply_dir, 'Supply_LSTM_y_scaler.pkl'))

print('========================================')
print('MODELS AND SCALERS LOADED SUCCESSFULLY')
print('========================================')
print('Demand Model :', 'Demand_LSTM_Final.keras')
print('Supply Model :', 'Supply_LSTM.keras')

MODELS AND SCALERS LOADED SUCCESSFULLY
Demand Model : Demand_LSTM_Final.keras
Supply Model : Supply_LSTM.keras


In [2]:
# ============================================================
# STEP 2: CALCULATE HISTORICAL RESIDUAL UNCERTAINTY (UP TO DEC 2025)
# ============================================================

d_X_train = joblib.load(os.path.join(demand_dir, 'Demand_X_train.pkl'))
d_X_test = joblib.load(os.path.join(demand_dir, 'Demand_X_test.pkl'))
d_y_test = joblib.load(os.path.join(demand_dir, 'Demand_y_test.pkl'))

d_X_tr_sc = d_X_scaler.transform(d_X_train.values)
d_X_te_sc = d_X_scaler.transform(d_X_test.values)
d_comb = np.vstack([d_X_tr_sc[-3:], d_X_te_sc])
d_X_seq = np.array([d_comb[i-3:i] for i in range(3, len(d_comb))])
d_test_pred = d_y_scaler.inverse_transform(d_model.predict(d_X_seq, verbose=0)).flatten()
d_test_act = d_y_test.values.flatten()
d_sigma = np.std(d_test_act - d_test_pred, ddof=1)

s_X_train = joblib.load(os.path.join(supply_dir, 'Supply_X_train.pkl'))
s_X_test = joblib.load(os.path.join(supply_dir, 'Supply_X_test.pkl'))
s_y_test = joblib.load(os.path.join(supply_dir, 'Supply_y_test.pkl'))

s_X_tr_sc = s_X_scaler.transform(s_X_train.values)
s_X_te_sc = s_X_scaler.transform(s_X_test.values)
s_comb = np.vstack([s_X_tr_sc[-6:], s_X_te_sc])
s_X_seq = np.array([s_comb[i-6:i] for i in range(6, len(s_comb))])
s_test_pred = s_y_scaler.inverse_transform(s_model.predict(s_X_seq, verbose=0)).flatten()
s_test_act = s_y_test.values.flatten()
s_sigma = np.std(s_test_act - s_test_pred, ddof=1)

gap_sigma = np.sqrt(d_sigma**2 + s_sigma**2)

print('========================================')
print('HISTORICAL VALIDATION UNCERTAINTY (UP TO DEC 2025)')
print('========================================')
print(f'Demand Standard Error (sigma_D) : {d_sigma:.2f} MU')
print(f'Supply Standard Error (sigma_S) : {s_sigma:.2f} MU')
print(f'Combined Gap Std Error (sigma_G): {gap_sigma:.2f} MU')

HISTORICAL VALIDATION UNCERTAINTY (UP TO DEC 2025)
Demand Standard Error (sigma_D) : 391.30 MU
Supply Standard Error (sigma_S) : 996.83 MU
Combined Gap Std Error (sigma_G): 1070.88 MU


In [3]:
# ============================================================
# STEP 3: DEMAND FORECAST PREDICTION INTERVALS TABLE
# ============================================================

months = ['Jan 2026', 'Feb 2026', 'Mar 2026']
z90, z95, z99 = norm.ppf(0.95), norm.ppf(0.975), norm.ppf(0.995)

d_rows = []
for i in range(3):
    p = d_preds[i]
    d_rows.append({
        'Month': months[i],
        'Actual Demand (MU)': f'{d_acts[i]:.2f}',
        'Predicted Demand (MU)': f'{p:.2f}',
        '90% Lower': f'{p - z90*d_sigma:.2f}',
        '90% Upper': f'{p + z90*d_sigma:.2f}',
        '95% Lower': f'{p - z95*d_sigma:.2f}',
        '95% Upper': f'{p + z95*d_sigma:.2f}',
        '99% Lower': f'{p - z99*d_sigma:.2f}',
        '99% Upper': f'{p + z99*d_sigma:.2f}'
    })

df_d_res = pd.DataFrame(d_rows)
print('====================================================================================================')
print('DEMAND FORECAST PREDICTION INTERVALS (JAN - MAR 2026)')
print('====================================================================================================')
print(df_d_res.to_string(index=False))
print('====================================================================================================')

DEMAND FORECAST PREDICTION INTERVALS (JAN - MAR 2026)
Month	Actual (MU)	Predicted (MU)	90% Lower	90% Upper	95% Lower	95% Upper	99% Lower	99% Upper
----------------------------------------------------------------------------------------------------
Jan 2026	10067.00	11047.51	10403.87	11691.15	10280.57	11814.46	10039.58	12055.45
Feb 2026	10125.00	12308.25	11664.62	12951.89	11541.31	13075.20	11300.32	13316.19
Mar 2026	12233.00	12594.89	11951.25	13238.53	11827.95	13361.83	11586.96	13602.82


In [4]:
# ============================================================
# STEP 4: SUPPLY FORECAST PREDICTION INTERVALS TABLE
# ============================================================

s_rows = []
for i in range(3):
    p = s_preds[i]
    s_rows.append({
        'Month': months[i],
        'Actual Supply (MU)': f'{s_acts[i]:.2f}',
        'Predicted Supply (MU)': f'{p:.2f}',
        '90% Lower': f'{p - z90*s_sigma:.2f}',
        '90% Upper': f'{p + z90*s_sigma:.2f}',
        '95% Lower': f'{p - z95*s_sigma:.2f}',
        '95% Upper': f'{p + z95*s_sigma:.2f}',
        '99% Lower': f'{p - z99*s_sigma:.2f}',
        '99% Upper': f'{p + z99*s_sigma:.2f}'
    })

df_s_res = pd.DataFrame(s_rows)
print('====================================================================================================')
print('SUPPLY FORECAST PREDICTION INTERVALS (JAN - MAR 2026)')
print('====================================================================================================')
print(df_s_res.to_string(index=False))
print('====================================================================================================')

SUPPLY FORECAST PREDICTION INTERVALS (JAN - MAR 2026)
Month	Actual (MU)	Predicted (MU)	90% Lower	90% Upper	95% Lower	95% Upper	99% Lower	99% Upper
----------------------------------------------------------------------------------------------------
Jan 2026	10189.56	8809.63	7170.00	10449.27	6855.89	10763.38	6241.97	11377.29
Feb 2026	10405.60	9229.95	7590.32	10869.59	7276.21	11183.70	6662.29	11797.61
Mar 2026	11247.51	9574.64	7935.00	11214.27	7620.89	11528.39	7006.98	12142.30


In [5]:
# ============================================================
# STEP 5: DEMAND - SUPPLY GAP PREDICTION INTERVALS TABLE
# ============================================================

g_rows = []
for i in range(3):
    act_gap = d_acts[i] - s_acts[i]
    pred_gap = d_preds[i] - s_preds[i]
    status = 'Shortage' if pred_gap > 0 else ('Surplus' if pred_gap < 0 else 'Balanced')
    g_rows.append({
        'Month': months[i],
        'Actual Gap (MU)': f'{act_gap:.2f}',
        'Predicted Gap (MU)': f'{pred_gap:.2f}',
        '90% Lower': f'{pred_gap - z90*gap_sigma:.2f}',
        '90% Upper': f'{pred_gap + z90*gap_sigma:.2f}',
        '95% Lower': f'{pred_gap - z95*gap_sigma:.2f}',
        '95% Upper': f'{pred_gap + z95*gap_sigma:.2f}',
        '99% Lower': f'{pred_gap - z99*gap_sigma:.2f}',
        '99% Upper': f'{pred_gap + z99*gap_sigma:.2f}',
        'Classification': status
    })

df_g_res = pd.DataFrame(g_rows)
print('================================================================================================------------------------')
print('DEMAND - SUPPLY GAP PREDICTION INTERVALS & CLASSIFICATION (JAN - MAR 2026)')
print('================================================================================================------------------------')
print(df_g_res.to_string(index=False))
print('================================================================================================------------------------')

================================================================================================------------------------
DEMAND - SUPPLY GAP PREDICTION INTERVALS & CLASSIFICATION (JAN - MAR 2026)
================================================================================================------------------------
Month	Actual Gap	Predicted Gap	90% Lower	90% Upper	95% Lower	95% Upper	99% Lower	99% Upper	Classification
------------------------------------------------------------------------------------------------------------------------
Jan 2026	-122.56	2237.88	476.44	3999.32	138.99	4336.77	-520.53	4996.29	Shortage
Feb 2026	-280.60	3078.30	1316.86	4839.74	979.41	5177.19	319.90	5836.71	Shortage
Mar 2026	985.49	3020.25	1258.81	4781.70	921.36	5119.14	261.85	5778.66	Shortage
================================================================================================------------------------
